In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Groq key loaded:", bool(GROQ_API_KEY))
print("Tavily key loaded:", bool(TAVILY_API_KEY))
print("OpenAI key loaded:", bool(OPENAI_API_KEY))

Groq key loaded: True
Tavily key loaded: True
OpenAI key loaded: True


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = DirectoryLoader("crew_data/", glob="*.txt", loader_cls=TextLoader)
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk preview:")
print(chunks[0].page_content)

Loaded 1 document(s)
Total chunks created: 4

First chunk preview:
CrewAI is a framework for orchestrating role-playing autonomous AI agents.
It enables agents to work together as a crew to accomplish complex tasks.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

print("FAISS vectorstore created successfully!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6827.01it/s]


FAISS vectorstore created successfully!
Total vectors stored: 4


In [4]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=TAVILY_API_KEY)

results = tavily.search(query="latest advancements in LLMs 2024")

for r in results["results"][:3]:
    print("Title:", r["title"])
    print("URL:", r["url"])
    print("Summary:", r["content"][:200])
    print("---")

Title: 2024 LLM Advancements and Trends | PDF | Deep Learning - Scribd
URL: https://www.scribd.com/document/887752145/Global-Advances-in-Large-Language-Models-in-2024
Summary: In 2024, large language models (LLMs) became essential infrastructure with significant advancements in capabilities, including reasoning-centric architectures
---
Title: 18 Artificial Intelligence LLM Trends in 2025 | by Gianpiero Andrenacci
URL: https://medium.com/data-bistrot/15-artificial-intelligence-llm-trends-in-2024-618a058c9fdf
Summary: LLM fine-tuning in 2024 is advancing through techniques like zero-shot and few-shot learning, enabling models to handle tasks with minimal or no
---
Title: The Evolving Landscape of Large Language Models (2024-2025)
URL: https://www.linkedin.com/pulse/evolving-landscape-large-language-models-2024-2025-zqgue
Summary: The global market for LLMs is experiencing exponential growth, valued at $6.4 billion in 2024 and projected to reach $36.1 billion by 2030,
---


In [ ]:
from langchain.tools import Tool

tavily_tool = Tool(
    name="WebSearch",
    description="Search the web for current information on any topic",
    func=lambda q: str(tavily.search(query=q)["results"][:2])
)


result = tavily_tool.run("What is CrewAI?")
print(result)

[{'url': 'https://www.digitalocean.com/resources/articles/what-is-crew-ai', 'title': 'What is CrewAI? A Platform to Build Collaborative AI Agents', 'content': 'A Platform to Build Collaborative AI Agents. Machine learning frameworks come and go, but CrewAI has emerged as a standout solution for a reason: it focuses on getting AI agents to work together rather than alone. CrewAI is an open-source project for developers that want to step beyond the limitations of solo AI agents. CrewAI is an open-source Python framework that organizes multiple AI agents into collaborative teams with specialized roles, making it more effective for complex tasks than single-agent approaches. CrewAI works best for complex use cases like automated research, content pipelines, and business intelligence, though teams seeking faster deployment might prefer alternatives like DigitalOcean’s Gradient Platform, Vertex AI, or LangChain, depending on their technical needs. CrewAI is an open-source Python framework th

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from pydantic import Field
import os

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


class TavilySearchTool(BaseTool):
    name: str = "WebSearch"
    description: str = "Search the web for current information on any topic"

    def _run(self, query: str) -> str:
        results = tavily.search(query=query)["results"][:2]
        return str(results)

tavily_crewai_tool = TavilySearchTool()

# Agent 1 — Researcher
researcher = Agent(
    role="Researcher",
    goal="Find detailed information about the given topic using web search",
    backstory="You are an expert researcher who finds accurate and relevant information.",
    tools=[tavily_crewai_tool],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

# Agent 2 — Writer
writer = Agent(
    role="Writer",
    goal="Write a clear and structured response based on research findings",
    backstory="You are a skilled writer who turns research into easy-to-read content.",
    tools=[],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

# Agent 3 — Critic
critic = Agent(
    role="Critic",
    goal="Review the written response for accuracy and completeness",
    backstory="You are a thorough reviewer who checks quality and suggests improvements.",
    tools=[],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

print("All 3 agents created successfully!")

All 3 agents created successfully!
